# Regularized Logistic Regression — Reusable Template

**Short name:** `WineReg` template. Swap the CSV, the target name, and the C windows.

Use this when the job is: *binary outcome, numeric (or already-encoded) features, compare unregularized vs L2 vs L1, tune C, drop dead weights.*


## Inline cheat-sheet (keep this cell visible)

See also **`WineReg_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Scale | `StandardScaler().fit(features)` then `.transform(features)` |
| Split | `train_test_split(X, y, test_size=0.2, random_state=99)` |
| No penalty | `LogisticRegression(penalty=None, max_iter=2000)` (lesson: `'none'`; future: `C=np.inf`) |
| Default ridge | `LogisticRegression()` historically L2 with `C=1` |
| F1 | `f1_score(y_true, y_pred)` — harmonic mean of precision & recall |
| Coarse C | `[0.0001, 0.001, 0.01, 0.1, 1]` — smaller C = stronger L2 |
| Fine C | `np.logspace(-4, -2, 100)` |
| Grid search | `GridSearchCV(clf, {'C': C_array}, scoring='f1', cv=5)` |
| L1 CV | `LogisticRegressionCV(Cs=..., penalty='l1', solver='liblinear', scoring='f1')` |
| Coefs | `pd.Series(clf.coef_.ravel(), predictors).sort_values()` |
| C meaning | `C = 1 / λ`. Tiny C → shrink toward 0. Huge C → unregularized. |

**sklearn ≥1.8 note.** `penalty` is deprecated. Lesson code with `penalty='none'` still runs; prefer `penalty=None` or `C=np.inf` for no regularization, and set `penalty='l2'` / `'l1'` explicitly in this notebook so the intent stays readable.


## Drop-in script

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import f1_score, roc_auc_score

# --- project knobs ---
DATA_PATH = "data/wine_quality.csv"
TARGET = "quality"
TEST_SIZE = 0.2
SEED = 99
COARSE_C = [0.0001, 0.001, 0.01, 0.1, 1]
FINE_C_LOG = (-4, -2)     # log10 window for ridge GridSearchCV
L1_C_LOG = (-2, 2)        # log10 window for Lasso CV
CV = 5
# ---------------------

df = pd.read_csv(DATA_PATH)
y = df[TARGET]
features = df.drop(columns=[TARGET])
X = StandardScaler().fit_transform(features)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED
)

unreg = LogisticRegression(penalty=None, max_iter=2000).fit(X_train, y_train)
print("unreg F1", f1_score(y_test, unreg.predict(X_test)))

rows = []
for C in COARSE_C:
    m = LogisticRegression(C=C, max_iter=2000).fit(X_train, y_train)
    rows.append((C, f1_score(y_train, m.predict(X_train)), f1_score(y_test, m.predict(X_test))))
print(pd.DataFrame(rows, columns=["C", "train_F1", "test_F1"]))

gs = GridSearchCV(
    LogisticRegression(max_iter=2000),
    param_grid={"C": np.logspace(*FINE_C_LOG, 80)},
    scoring="f1", cv=CV,
).fit(X_train, y_train)
print("best ridge", gs.best_params_, gs.best_score_)

l1 = LogisticRegressionCV(
    Cs=np.logspace(*L1_C_LOG, 80), cv=CV, penalty="l1",
    solver="liblinear", scoring="f1", max_iter=2000,
).fit(X, y)
print("best L1 C", l1.C_)
print(pd.Series(l1.coef_.ravel(), index=features.columns).sort_values())


## How to reuse

1. Point `DATA_PATH` / `TARGET` at the new table.
2. Dummy-encode categoricals *before* scaling.
3. Redraw the coarse C grid if the plot is monotone — the wine window is not universal.
4. Keep a frozen test fold out of `GridSearchCV`.
5. Read L1 zeros as “candidates to drop”, then confirm with a refit on the reduced card.
6. Rewrite the four audience paragraphs; do not ship the wine wording to a credit-risk book.

**Good fit:** binary flag, n in the hundreds to tens of thousands, mostly numeric lab / sensor / scorecard inputs.  
**Bad fit:** rare events with <10 events per weight, raw 1–10 ordinal targets, price prediction, causal “this chemical *makes* the wine good”.
